Code taken from:
https://umap-learn.readthedocs.io/en/latest/basic_usage.html

Install the umap module with conda:
`conda install -c conda-forge umap-learn`

or with pip: 
`pip install umap-learn`

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
%matplotlib inline
import keras

import umap

In [ ]:
import json, sys
from scipy.io import  wavfile
from IPython import display

import str_ww_util as util
import get_dataset
import keras_model as models

from IPython import display


In [ ]:
sns.set(style='white', context='notebook', rc={'figure.figsize':(14,10)})

In [ ]:
digits = load_digits()
digits_df = pd.DataFrame(digits.data[:,1:11])
digits_df['digit'] = pd.Series(digits.target).map(lambda x: 'Digit {}'.format(x))


In [ ]:
rng = np.random.default_rng(2024)

In [ ]:
from get_dataset import get_data, get_file_lists, get_data_config

In [ ]:
# jupyter will pass an extra -f=<tmp_file> arg, which throws an 
# unrecognized argument error
sys.argv = sys.argv[0:1] 

Flags = util.parse_command()

In [ ]:
notebook_mode = "short_inference" # "short_inference" OR "inference" OR "short_training" OR "full_training"

if notebook_mode == "inference": 
  load_pretrained_model = True
  save_model = False
elif notebook_mode == "short_inference": 
  load_pretrained_model = True
  save_model = False
  Flags.num_samples_training = 2000
  Flags.num_samples_validation = 1000
  Flags.num_samples_test = 1000
else:
  # Or make custom settings here
  pass

# 'trained_models/str_ww_model.h5' is the default save path for train.py
# pretrained_model_path = 'trained_models/str_ww_ref_model.h5' # path to load from if load_pretrained_model is True
pretrained_model_path = 'trained_models/str_ww_model.h5' # path to load from if load_pretrained_model is True

samp_freq = Flags.sample_rate
label_list=['marvin', 'silent', 'other']

In [ ]:
try:
    with open('streaming_config.json', 'r') as fpi:
        streaming_config = json.load(fpi)
    Flags.data_dir = streaming_config['speech_commands_path']
except:
    raise RuntimeError("""
        In this directory, copy streaming_config_template.json to streaming_config.json
        and edit it to point to the directories where you have the speech commands dataset
        and (optionally) the MUSAN noise data set.
        """)
Flags.bg_path = Flags.data_dir

In [ ]:
ds_train, ds_test, ds_val = get_dataset.get_all_datasets(Flags)

In [ ]:
x1, y1 = ds_train.as_numpy_iterator().next()
x_shape = x1.shape[1:]
y_shape = y1.shape[1:]
x_train = np.zeros((0,)+x_shape)
y_train = np.zeros((0,)+y_shape)
for x,y in ds_train:
  x_train = np.concatenate((x_train,x), axis=0)
  y_train = np.concatenate((y_train,y), axis=0)

y_train = np.argmax(y_train, axis=1) # convert from one hot to index

In [ ]:
print(f"Loading pretrained model from {pretrained_model_path}")
model = keras.models.load_model(pretrained_model_path)
model.summary()

In [ ]:
use_features = True
feature_layer = -4
if use_features:
  feature_model = keras.Model(inputs=model.input, outputs=model.layers[feature_layer].output)
  feature_data = feature_model(x_train)
  feature_model.summary()  
  feature_data = feature_data.numpy().reshape(feature_data.shape[0], -1)
  print(f"Using layer {feature_layer}, {model.layers[feature_layer].name},",
        f"type {model.layers[feature_layer].__class__.__name__}")
else:
  print(f"Using input spectrograms")
  feature_data = x_train.squeeze()
  feature_data = feature_data.reshape(x_train.shape[0], -1)

In [ ]:
reducer = umap.UMAP(random_state=42)
reducer.fit(feature_data)

In [ ]:
embedding = reducer.transform(feature_data)
# Verify that the result of calling transform is
# idenitical to accessing the embedding_ attribute
assert(np.all(embedding == reducer.embedding_))
embedding.shape

In [ ]:
plt.figure(figsize=(8,8))
num_samples = x_train.shape[0]
idx = slice(0,num_samples)
plt.scatter(embedding[idx, 0], embedding[idx, 1], c=y_train[idx], cmap='Spectral', s=15)
plt.gca().set_aspect('equal', 'datalim')
hCBar = plt.colorbar(boundaries=np.arange(4)-0.5)
hCBar.set_ticks(np.arange(3))
hCBar.set_ticklabels(label_list)
plt.title('UMAP projection of the Keywords', fontsize=24);

In [ ]:
1/0

In [ ]:
from io import BytesIO
from PIL import Image
import base64

In [ ]:
def embeddable_image(data):
    img_data = 255 - 15 * data.astype(np.uint8)
    image = Image.fromarray(img_data, mode='L').resize((64, 64), Image.BICUBIC)
    buffer = BytesIO()
    image.save(buffer, format='png')
    for_encoding = buffer.getvalue()
    return 'data:image/png;base64,' + base64.b64encode(for_encoding).decode()

In [ ]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool, ColumnDataSource, CategoricalColorMapper
from bokeh.palettes import Spectral10

In [ ]:
# notebook output does not work with the zoom tools for me, so let it render on a separate tab
output_notebook() 

In [ ]:
digits_df = pd.DataFrame(embedding, columns=('x', 'y'))
digits_df['digit'] = [str(x) for x in digits.target]
digits_df['image'] = list(map(embeddable_image, digits.images))

datasource = ColumnDataSource(digits_df)
color_mapping = CategoricalColorMapper(factors=[str(9 - x) for x in digits.target_names],
                                       palette=Spectral10)

plot_figure = figure(
    title='UMAP projection of the Digits dataset',
    width=600, # was plot_width=600,
    height=600, # was plot_height=600, 
    tools=(['pan', 'reset','wheel_zoom'])  # can replace wheel_zoom with zoom_in, zoom_out
)

plot_figure.add_tools(HoverTool(tooltips="""
<div>
    <div>
        <img src='@image' style='float: left; margin: 5px 5px 5px 5px'/>
    </div>
    <div>
        <span style='font-size: 16px; color: #224499'>Digit:</span>
        <span style='font-size: 18px'>@digit</span>
    </div>
</div>
"""))

plot_figure.scatter(
    'x',
    'y',
    source=datasource,
    color=dict(field='digit', transform=color_mapping),
    line_alpha=0.6,
    fill_alpha=0.6,
    size=4
)
show(plot_figure)

In [ ]:
# This code works fine in by itself, but if I run it here, it gets the same error as 
# above, with the zoom tools not having renderers.
from bokeh.plotting import figure, show

p = figure(width=400, height=400,
           title=None, toolbar_location="below")

p.scatter([1, 2, 3, 4, 5], [2, 5, 8, 2, 7], size=10)

show(p)